In [22]:
import os
import yaml
import pytz
import requests
import numpy as np
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

# import functions from bookie_grabber
from bookie_grabber import *

# import betfair_api funcs
from betfair_api import *

load_dotenv()

True

In [23]:
session_token = get_session_token()
print("SESSION TOKEN:", session_token)

# Pick a league from config (for example, first league)
league_name = LEAGUES[0]["name"]
df = get_ou_volume(session_token, league_name)
print(df.head())

SESSION TOKEN: +Nl/mIRoq2CNFD8aMcuUcp8Pp9vzsgYoPxbrhPjZS8Y=
      marketId                  line  total_volume                  event  \
0  1.250884917  Over/Under 3.5 Goals  13947.824304     Man Utd v West Ham   
1  1.250884904  Over/Under 2.5 Goals  27693.932352     Man Utd v West Ham   
2  1.250884908  Over/Under 1.5 Goals  20228.228208     Man Utd v West Ham   
3  1.251156061  Over/Under 2.5 Goals   1567.433520  Aston Villa v Arsenal   
4  1.251156074  Over/Under 3.5 Goals     78.932448  Aston Villa v Arsenal   

                    kickoff  
0  2025-12-04T20:00:00.000Z  
1  2025-12-04T20:00:00.000Z  
2  2025-12-04T20:00:00.000Z  
3  2025-12-06T12:30:00.000Z  
4  2025-12-06T12:30:00.000Z  


In [24]:
df

,marketId,line,total_volume,event,kickoff
0,1.250884917,Over/Under 3.5 Goals,13947.824304,Man Utd v West Ham,2025-12-04T20:00:00.000Z
1,1.250884904,Over/Under 2.5 Goals,27693.932352,Man Utd v West Ham,2025-12-04T20:00:00.000Z
2,1.250884908,Over/Under 1.5 Goals,20228.228208,Man Utd v West Ham,2025-12-04T20:00:00.000Z
3,1.251156061,Over/Under 2.5 Goals,1567.433520,Aston Villa v Arsenal,2025-12-06T12:30:00.000Z
4,1.251156074,Over/Under 3.5 Goals,78.932448,Aston Villa v Arsenal,2025-12-06T12:30:00.000Z
5,1.251156065,Over/Under 1.5 Goals,0.000000,Aston Villa v Arsenal,2025-12-06T12:30:00.000Z
6,1.251155787,Over/Under 1.5 Goals,0.000000,Man City v Sunderland,2025-12-06T15:00:00.000Z
7,1.251157047,Over/Under 3.5 Goals,147.437568,Tottenham v Brentford,2025-12-06T15:00:00.000Z
8,1.251155783,Over/Under 2.5 Goals,117.448896,Man City v Sunderland,2025-12-06T15:00:00.000Z
9,1.251157038,Over/Under 1.5 Goals,593.387712,Tottenham v Brentford,2025-12-06T15:00:00.000Z


In [ ]:
events = get_league_events(api_key)
df_events = extract_events_to_df(events)


perth_tz = pytz.timezone("Australia/Perth")
now = datetime.now(perth_tz)

# Only include matches within ±30 minutes of exactly 9 hours from KO
df_events = df_events[
    (df_events["match_time"] - now).dt.total_seconds().between(9*3600 - 1800, 9*3600 + 1800)
].reset_index(drop=True)

print(f"Filtered to {len(df_events)} matches ~9 hours before KO.")

In [ ]:
# ┌─────────────────────────────────────────────────────────┐
# │  1. SPORTS                                              │
# │  What sports are available?                             │
# │  GET /v3/sports                                         │
# │  Returns: Football, Basketball, Tennis, etc.            │
# └──────────────────┬──────────────────────────────────────┘
#                    │
#                    ↓
# ┌─────────────────────────────────────────────────────────┐
# │  2. LEAGUES                                             │
# │  What leagues exist in a sport?                         │
# │  GET /v3/leagues?sport=football                         │
# │  Returns: Premier League, La Liga, Bundesliga, etc.     │
# └──────────────────┬──────────────────────────────────────┘
#                    │
#                    ↓
# ┌─────────────────────────────────────────────────────────┐
# │  3. EVENTS                                              │
# │  What matches are coming up?                            │
# │  GET /v3/events?sport=football&league=premier-league    │
# │  Returns: Man Utd vs Liverpool, Chelsea vs Arsenal, etc.│
# └──────────────────┬──────────────────────────────────────┘
#                    │
#                    ↓
# ┌─────────────────────────────────────────────────────────┐
# │  4. ODDS                                                │
# │  What are the betting odds?                             │
# │  GET /v3/odds?eventId=123456&bookmakers=Bet365,Unibet   │
# │  Returns: Odds from multiple bookmakers                 │
# └─────────────────────────────────────────────────────────┘

In [ ]:
# "status": "pending" for events
# Example usage
df_totals, df_btts = main()

In [ ]:
df = pd.read_csv("/Users/notbahd/Desktop/BookieGrabber/data/exports/totals/totals_hdp_2.5_20251123.csv")
df

In [ ]:
# Hey mate, the volume is already available, listed under “depth” in the markets array.

In [ ]:
api_key = load_env()
odds = get_event_odds(api_key, 61300801)

In [ ]:
odds.keys()

In [ ]:
for bookmaker, markets in odds.get("bookmakers", {}).items():
    for m in markets:
        print(f"{bookmaker}- {m.get('name')}" )

In [ ]:
btts = []
for bookmaker, markets in odds.get("bookmakers", {}).items():
    for market in markets:
        if market.get("name") == "Both Teams to Score" or market.get("name") == "Both Teams To Score":
            for o in market.get("odds", []):
                    btts.append({
                        "bookmaker": bookmaker,
                        "yes": o.get("yes"),
                        "no": o.get("no"),
                    })
btts

In [ ]:
btts